# TopoGen Earth — Flow Matching Demo (OpenEarthMap)

Satellite image generation conditioned on land-cover masks.

Uses code from `src/` — clone the repo alongside this notebook:
```
!git clone https://github.com/mihalko711/topogen-earth.git
%cd topogen-earth
```

In [ ]:
import os
import sys
sys.path.insert(0, "..")  # if running from notebooks/ subdir

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from src.data.dataset import OpenEarthMapDataset
from src.models.config import UNetConfig, TrainingConfig
from src.models.model import create_unet, generate_steps
from src.training.trainer import Trainer

In [ ]:
DATA_PATH = (
    "/kaggle/input/aletbm/global-land-cover-mapping-openearthmap"
    if os.path.exists("/kaggle/input")
    else ""
)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"DATA_PATH: {DATA_PATH!r}")

## 1. Dataset demo

In [ ]:
dataset = OpenEarthMapDataset(
    root_dir=DATA_PATH,
    split="train",
    crop_size=128,
    subset_size=100,
)

print(f"Dataset size: {len(dataset)}")
sample = dataset[0]
print(f"Image shape: {sample['image'].shape}")
print(f"Mask shape:  {sample['mask'].shape}")

In [ ]:
def denormalize(t):
    return (t.permute(1, 2, 0) * 0.5 + 0.5).clamp(0, 1)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    s = dataset[i]
    axes[0, i].imshow(denormalize(s["image"]))
    axes[0, i].set_title(f"Image {i}")
    axes[0, i].axis("off")
    axes[1, i].imshow(denormalize(s["mask"]))
    axes[1, i].set_title(f"Mask {i}")
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()

## 2. Model

In [ ]:
model_cfg = UNetConfig(
    sample_size=128,
    in_channels=6,
    out_channels=3,
    layers_per_block=2,
    block_out_channels=(32, 64, 128),
)
model = create_unet(model_cfg).to(device)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

## 3. Training (quick demo — 5 epochs)

In [ ]:
train_cfg = TrainingConfig(
    num_epochs=5,
    batch_size=16,
    learning_rate=1e-4,
    crop_size=128,
    save_dir="experiments_demo",
    viz_interval=5,
    num_steps_generation=8,
)

dataloader = DataLoader(
    dataset,
    batch_size=train_cfg.batch_size,
    shuffle=True,
    num_workers=train_cfg.num_workers,
)

trainer = Trainer(model, dataloader, train_cfg, device=device)
trainer.run()

## 4. Inference on a random mask

In [ ]:
model.eval()
idx = np.random.randint(len(dataset))
sample = dataset[idx]

mask = sample["mask"]
target = sample["image"]

history = generate_steps(
    model, mask, num_steps=10, device=device, crop_size=128
)

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
axes[0, 0].imshow(denormalize(mask))
axes[0, 0].set_title("Input mask")
axes[0, 0].axis("off")
axes[0, 1].imshow(denormalize(target))
axes[0, 1].set_title("Target")
axes[0, 1].axis("off")

for i, h in enumerate(history[:8]):
    r = (i + 2) // 6
    c = (i + 2) % 6
    axes[r, c].imshow(denormalize(h.squeeze(0)))
    axes[r, c].set_title(f"t={(i)/(len(history)-1):.1f}")
    axes[r, c].axis("off")

for i in range(8, 10):
    r = (i + 2) // 6
    c = (i + 2) % 6
    axes[r, c].axis("off")

plt.tight_layout()
plt.show()